In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.bronze")

In [0]:
bucket = "s3://ro-company-lake"
onrc_nomenclatoare_path = f"{bucket}/raw_v2/onrc/nomenclatoare/"

display(dbutils.fs.ls(onrc_nomenclatoare_path))

In [0]:
def list_all_files(path):
    all_files = []

    def recurse(p):
        for item in dbutils.fs.ls(p):
            if item.isDir():
                recurse(item.path)
            else:
                all_files.append(item.path)

    recurse(path)
    return all_files


all_files = list_all_files(onrc_nomenclatoare_path)

n_caen_files = [
    f for f in all_files
    if f.lower().endswith("/n_caen.csv") or f.lower().endswith("n_caen.csv")
]

print("n_caen files found:")
for f in n_caen_files:
    print(f)

print("Count:", len(n_caen_files))

if len(n_caen_files) == 0:
    raise ValueError("No n_caen.csv files found in S3.")

In [0]:
n_caen_path = n_caen_files[0]

print("Using:", n_caen_path)

n_caen_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "^")
    .csv(n_caen_path)
)

display(n_caen_raw.limit(50))
print("Rows:", n_caen_raw.count())
print("Columns:", n_caen_raw.columns)

In [0]:
(
    n_caen_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.bronze.n_caen_raw")
)

In [0]:
display(spark.sql("""
SELECT *
FROM company_ro.bronze.n_caen_raw
LIMIT 100
"""))

print(spark.table("company_ro.bronze.n_caen_raw").columns)